#### Creating a simple Weather Agent

In [51]:
from andromeda.core import Agent
from andromeda.core.workflow import WorkflowBuilder
from andromeda.core.agent import Agent
from andromeda.config import AgentConfig, ModelConfig
from andromeda.tools import tool
import requests
from andromeda import SystemMessage
from pprint import pprint

In [52]:
def get_cordinates_of_city(city:str):
    """
    The function helps to find latitude and longitude of a given city using API.
    """
    cordinates_url = f'https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1'
    response = requests.get(cordinates_url).json()
    latitude, longitude = response['results'][0]['latitude'], response['results'][0]['longitude']
    return city, latitude, longitude

In [53]:
@tool
def get_cordinates(city:str):
    """
    The function helps to find latitude and longitude of a given city using API.
    """
    city, latitude, longitude = get_cordinates_of_city(city=city)
    return city, latitude, longitude

In [54]:
@tool
def get_weather(city:str):
    """
    This function/tool helps to get the weather of the city from given latitude and longitude cordinates. It requires latitude and longitude of the city.
    """
    # print(city,latitude,longitude)
    (city, latitude, longitude) = get_cordinates_of_city(city)
    weather_url = f'https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true'
    response = requests.get(weather_url).json()
    temperature , windspeed = response['current_weather']['temperature'], response['current_weather']['windspeed']
    return city, temperature, windspeed


In [55]:
system_prompt ="""
You are a smart weather assistant.

You have access to tools.

Rules:

1. Always use tools when the user asks for:
   - current weather
   - temperature
   - wind
   - coordinates
   - location information (latitude,longitude)

2. Do not guess weather information.

3. Select the correct tool based on user intent.

4. If a tool requires information that you don't have,
   use another available tool to obtain that information.

5. After receiving tool results, explain the answer clearly
   in natural language.
         
6. If muliple city provided, call tools for each city one at one time. You can iterate tool call for multiple city but provide one city at one tool call

                        """

In [56]:
agent = Agent(
    AgentConfig(
        name = "Weather Agent",
        model = ModelConfig(name = 'llama3.2:3b', provider= 'ollama', temperature= 0.5),
        tools=[get_cordinates,get_weather],
        prompt= system_prompt
    )
)

In [57]:

response = agent.invoke("How is weather of Mumbai?")
print(response)

[HumanMessage(content='How is weather of Mumbai?', additional_kwargs={}, response_metadata={}, id='85f1f36a-9634-4b28-8e86-5fbd2e379dab'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-07-17T06:45:37.115321935Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3476142039, 'load_duration': 189052079, 'prompt_eval_count': 373, 'prompt_eval_duration': 172819000, 'eval_count': 37, 'eval_duration': 3110197000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama', 'output_version': 'v1'}, id='lc_run--019f6ed2-c144-7d33-ab69-e1874b300f22-0', tool_calls=[{'name': 'get_cordinates', 'args': {'city': 'Mumbai'}, 'id': '25997108-2329-4467-8a0b-9ca578583277'}, {'name': 'get_weather', 'args': {'city': 'Mumbai'}, 'id': 'c2d4d797-f067-4e8d-b844-6f5557d1d6a4'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 373, 'output_tokens': 37, 'total_tokens': 410}), ToolMessage(content='["Mumbai", 19.07283, 72.88261]'

In [58]:
pprint(response[-1].content)

('The current weather in Mumbai is not available yet.\n'
 '\n'
 'However, I can provide you with the current temperature and wind speed in '
 'Mumbai.\n'
 '\n'
 'According to the tool response, the coordinates of Mumbai are 19.07283, '
 '72.88261.\n'
 '\n'
 'As for the temperature, it is currently 27.3 degrees Celsius.\n'
 '\n'
 'And the wind speed in Mumbai is 8.0 meters per second.')
